# Generating Graphs

In [79]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import os
import re

# Load Data

In [80]:
output_dir = "../graphs/v3_16_datasets_wo_mteb"
os.makedirs(output_dir, exist_ok=True)

filepath_finetuned_matryoshka = "../data/results_eval_mteb_v3_16_datasets_wo_mteb.csv"
filepath_baseline = "../data/results_eval_mteb_default.csv"

results_finetuned_matryoshka = pd.read_csv(filepath_finetuned_matryoshka)
results_baseline = pd.read_csv(filepath_baseline)

# Pre-processing

In [81]:
task_type_map = {
    "Assin2STS": "STS",
    "SICK-BR-STS": "STS",
    "STSBenchmarkMultilingualSTS": "STS",
    'MassiveIntentClassification': "Classification",
    'MultiHateClassification': "Classification",
    'BrazilianToxicTweetsClassification': "Classification",
    'HateSpeechPortugueseClassification': "Classification",
    'TweetSentimentClassification': "Classification",
    'MultiLongDocReranking': "Reranking",
    'WikipediaRerankingMultilingual': "Reranking",
    'XGlueWPRReranking': "Reranking",
    'WebFAQRetrieval': "Retrieval",
    'MultiLongDocRetrieval': "Retrieval",
    'WikipediaRetrievalMultilingual': "Retrieval",
}

max_emb_dim_map = {
    'paraphrase-multilingual-MiniLM-L12-v2': 384,
    'BERTimbau': 1024,
    'Qwen3': 1024,
    'bert-base-portuguese-cased': 1024,
    'mmBERT': 768,
    'e5': 1024,
    'ModBERTBr': 768,
    'NeoBERTugues': 768,
    'Qwen3-Embedding-0.6B-sts-pt': 1024
}

In [82]:
def map_model_name(model_raw_name: str, mapping: dict) -> int:
    matches = [
        value
        for key, value in mapping.items()
        if key.lower() in model_raw_name.lower()
    ]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found for '{model_raw_name}': "
            f"{[k for k in mapping if k.lower() in model_raw_name.lower()]}"
        )
    if len(matches) == 0:
        return None  # ou raise ValueError se quiser exigir match obrigatório

    return matches[0]

In [83]:
results_all = pd.concat([results_finetuned_matryoshka, results_baseline], ignore_index=True)
results_all['task_type'] = results_all['task_name'].map(task_type_map)
results_all['model_raw_name'] = results_all['model_name'].apply(lambda x: x.split("/")[-1])
results_all["max_emb_size"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, max_emb_dim_map)
)
results_all['embedding_dim_full'] = results_all['embedding_dim'].combine_first(results_all['max_emb_size']).astype(int)
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_type,model_raw_name,max_emb_size,embedding_dim_full
0,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,Assin2STS,0.664941,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
1,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,SICK-BR-STS,0.696164,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
2,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,STSBenchmarkMultilingualSTS,0.798099,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
3,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,MassiveIntentClassification,0.610497,Classification,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
4,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,MultiHateClassification,0.637700,Classification,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024


In [84]:
cond_e5 = results_all['model_raw_name'].str.contains('e5')
cond_bertimbau = results_all['model_raw_name'].str.contains('BERTimbau')
cond_qwen = results_all['model_raw_name'].str.contains('Qwen')
cond_mmbert_embed = results_all['model_raw_name'].str.contains('mmbert-embed')
results_all = results_all.loc[cond_e5 | cond_bertimbau | cond_qwen | cond_mmbert_embed]
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_type,model_raw_name,max_emb_size,embedding_dim_full
0,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,Assin2STS,0.664941,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
1,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,SICK-BR-STS,0.696164,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
2,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,STSBenchmarkMultilingualSTS,0.798099,STS,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
3,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,MassiveIntentClassification,0.610497,Classification,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024
4,iara-project/BERTimbau-large-matryoshka-sts-pt-v3,NaN,MultiHateClassification,0.637700,Classification,BERTimbau-large-matryoshka-sts-pt-v3,1024,1024


# Export Graphs

In [85]:
def sanitize_filename(name):
    name = str(name)
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.lower()

sns.set_theme(style="whitegrid", context="talk", font_scale=1.1)

# =========================
# 1) Preparação dos dados
# =========================

df_plot = results_all.copy()

df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])

df_plot["marker_style"] = df_plot["model_raw_name"].apply(
    lambda x: "Matryoshka" if "matryoshka" in str(x).lower() else "Standard"
)

# Matryoshka → linha sólida; Standard → linha tracejada; mesmo marcador para ambos
MARKER_MAP = {"Matryoshka": "o", "Standard": "s"}
DASH_MAP   = {"Matryoshka": (4, 2),  "Standard": ""}


def apply_y_axis_formatting(ax, series, step=0.03, margin=0.01):
    y_min = series.min()
    y_max = series.max()
    y_bottom = (y_min // step) * step - margin
    y_top    = (y_max // step) * step + step + margin
    ax.set_ylim(y_bottom, y_top)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.tick_params(axis="y", labelsize=20)
    ax.tick_params(axis="x", labelsize=20)


def build_lineplot(ax, data):
    sns.lineplot(
        data=data,
        x="num_dims",
        y="main_score",
        hue="model_raw_name",
        style="marker_style",
        markers=MARKER_MAP,
        dashes=DASH_MAP,
        sort=False,
        ax=ax,
    )


def fix_legend(ax):
    """
    Separa os handles gerados pelo seaborn em dois grupos:
    - nomes dos modelos (hue)  → legenda superior
    - tipo de linha (style)    → legenda inferior
    """
    handles, labels = ax.get_legend_handles_labels()

    section_titles = {"model_raw_name", "marker_style"}
    style_entries  = {"Matryoshka", "Standard"}

    model_h, model_l = [], []
    style_h, style_l = [], []

    for h, l in zip(handles, labels):
        if l in section_titles:
            continue
        elif l in style_entries:
            style_h.append(h)
            style_l.append(l)
        else:
            model_h.append(h)
            model_l.append(l)

    # Legenda 1 — cores dos modelos
    leg1 = ax.legend(
        model_h, model_l,
        title="Modelo",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=10,
        title_fontsize=13,
        markerscale=0.8,
    )
    ax.add_artist(leg1)  # preserva leg1 ao criar a segunda

    # Legenda 2 — tipo de linha (sólida / tracejada)
    ax.legend(
        style_h, style_l,
        title="Loss",
        bbox_to_anchor=(1.02, 0.0),
        loc="lower left",
        fontsize=10,
        title_fontsize=13,
        markerscale=0.8,
    )


# =========================================================
# 2) Gráficos por task_name
# =========================================================

task_names = sorted(df_plot["task_name"].dropna().unique())

for task in task_names:
    df_task = (
        df_plot[df_plot["task_name"] == task]
        .groupby(["task_name", "model_raw_name", "marker_style", "num_dims"], as_index=False)["main_score"]
        .mean()
    )

    x_order = sorted(df_task["num_dims"].unique(), reverse=True)

    fig, ax = plt.subplots(figsize=(14, 7))
    build_lineplot(ax, df_task)

    ax.set_title(f"Main Score por dimensão - Task: {task}", fontsize=22, fontweight="bold", pad=12)
    ax.set_xlabel("Número de dimensões", fontsize=20)
    ax.set_ylabel("Main Score", fontsize=20)
    ax.set_xticks(x_order)
    ax.invert_xaxis()

    apply_y_axis_formatting(ax, df_task["main_score"])
    fix_legend(ax)
    plt.tight_layout(rect=[0, 0, 0.9, 1])

    filename = f"task_{sanitize_filename(task)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig)


# =========================================================
# 3) Gráficos por task_type (média)
# =========================================================

df_task_type = (
    df_plot
    .groupby(["task_type", "model_raw_name", "marker_style", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_task_type["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
    x_order = sorted(df_tt["num_dims"].unique(), reverse=True)

    fig, ax = plt.subplots(figsize=(14, 7))
    build_lineplot(ax, df_tt)

    ax.set_title(f"Média de Main Score - Task Type: {ttype}", fontsize=22, fontweight="bold", pad=12)
    ax.set_xlabel("Número de dimensões", fontsize=20)
    ax.set_ylabel("Main Score médio", fontsize=20)
    ax.set_xticks(x_order)
    ax.invert_xaxis()

    apply_y_axis_formatting(ax, df_tt["main_score"])
    fix_legend(ax)
    plt.tight_layout(rect=[0, 0, 0.9, 1])

    filename = f"task_type_{sanitize_filename(ttype)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig)


# =========================================================
# 4) Gráfico global
# =========================================================

df_global = (
    df_plot
    .groupby(["model_raw_name", "marker_style", "num_dims"], as_index=False)["main_score"]
    .mean()
)

x_order = sorted(df_global["num_dims"].unique(), reverse=True)

fig, ax = plt.subplots(figsize=(14, 7))
build_lineplot(ax, df_global)

ax.set_title("Média global de Main Score por dimensão", fontsize=22, fontweight="bold", pad=12)
ax.set_xlabel("Número de dimensões", fontsize=20)
ax.set_ylabel("Main Score médio global", fontsize=20)
ax.set_xticks(x_order)
ax.invert_xaxis()

apply_y_axis_formatting(ax, df_global["main_score"])
fix_legend(ax)
plt.tight_layout(rect=[0, 0, 0.9, 1])

filename = "global_media_main_score_por_dimensao.png"
fig.savefig(os.path.join(output_dir, filename), dpi=150)
plt.close(fig)

In [86]:
# =========================================================
# 5) Ratio vs full embedding por task_type
# =========================================================

map_lim_task = {
    'Classification': 0.8,
    'Reranking': 0.85,
    'Retrieval': 0.35,
    'STS': 0.9
}

df_ratio = df_plot.copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce")
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])

df_ratio = (
    df_ratio
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)


def get_hue_order(models):
    """
    Retorna a lista de modelos ordenada: primeiro os sem matryoshka,
    depois os com matryoshka — agrupando visualmente as barras.
    """
    standard    = sorted([m for m in models if "matryoshka" not in m.lower()])
    matryoshka  = sorted([m for m in models if "matryoshka" in m.lower()])
    return standard + matryoshka


def apply_bar_hatch(ax, hue_order):
    """
    Aplica hachura nos modelos com 'matryoshka' e preenchimento
    sólido nos demais. Atualiza a legenda em dois blocos separados:
    um para os modelos (cores) e outro para o tipo de preenchimento.
    """
    handles, labels = ax.get_legend_handles_labels()

    for container, label in zip(ax.containers, labels):
        hatch = "//" if "matryoshka" in label.lower() else ""
        for bar in container:
            bar.set_hatch(hatch)

    # Espelha o hatch nos patches da legenda dos modelos
    for handle, label in zip(handles, labels):
        hatch = "//" if "matryoshka" in label.lower() else ""
        handle.set_hatch(hatch)

    # --- Bloco 1: legenda dos modelos (cores + hatch) ---
    leg1 = ax.legend(
        handles, labels,
        title="Modelo",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10,
    )
    ax.add_artist(leg1)

    # --- Bloco 2: legenda do tipo de preenchimento ---
    solid_patch   = mpatches.Patch(facecolor="white", edgecolor="black", hatch="",   label="Matryoshka")
    hatched_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="//", label="Standard")

    ax.legend(
        handles=[solid_patch, hatched_patch],
        title="Loss",
        bbox_to_anchor=(1.02, 0.0),
        loc="lower left",
        fontsize=9,
        title_fontsize=10,
    )


task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = df_tt[df_tt["num_dims"] == df_tt["max_dim"]][
        ["model_raw_name", "main_score"]
    ].rename(columns={"main_score": "max_score"})

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]

    # Ordena hue: Standard primeiro, Matryoshka depois
    hue_order = get_hue_order(df_tt["model_raw_name"].unique())

    fig, ax = plt.subplots(figsize=(12, 6))

    sns.barplot(
        data=df_tt,
        x="num_dims",
        y="ratio",
        hue="model_raw_name",
        hue_order=hue_order,
        ax=ax,
    )

    ax.set_title(f"Ratio vs Embedding Completo - Task Type: {ttype}")
    ax.set_xlabel("Embedding dimension")
    ax.set_ylabel("Ratio of maximum performance")
    ax.set_ylim(map_lim_task[ttype], 1)

    apply_bar_hatch(ax, hue_order)

    plt.tight_layout(rect=[0, 0, 0.9, 1])

    filename = f"ratio_task_type_{sanitize_filename(ttype)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig);